In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Matplotlib is building the font cache; this may take a moment.


In [ ]:
df = pd.read_csv("../data/raw/spotify-tracks-dataset-detailed.csv")
pd.set_option("display.max_columns", None)

In [ ]:
# dataset audit
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# missing value audit
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100
})
# sort missing values
missing = missing.sort_values("missing_pct", ascending=False)
missing

In [ ]:
# duplicate rows
duplicate_rows = df.duplicated().sum()
print(f"Duplicated rows: {duplicate_rows:,}")
# duplicate track IDs
duplicate_track_ids = df["track_id"].duplicated().sum()
print(f"Duplicate track IDs: {duplicate_track_ids:,}")

In [ ]:
# investigate duplicate IDs
duplicate_tracks = df[
    df["track_id"].duplicated(keep=False)
].sort_values("track_id")
duplicate_tracks.head(20)

In [ ]:
# check whether duplicated IDs contain conflicted values
if duplicate_track_ids > 0:
    consistency = duplicated_tracks.groupby("track_id").nunique()
    consistency
else:
    print("No duplicate track IDs found")

In [ ]:
# candidate features
audio_features = [
    "danceability",
    "energy",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "valence",
    "tempo",
    "loudness",
    "duration_ms",
    "key",
    "mode",
    "time_signature"
]

In [ ]:
# check feature dtypes
df[audio_features].dtypes

In [ ]:
# check non-numeric features
non_numeric = df[audio_features].select_dtypes(
    exclude=np.number
).columns
print("Non-numeric columns:", list(non_numeric))

In [ ]:
# check bounded features
bounded_features = [
    "danceability",
    "energy",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "valence"
]

for feature in bounded_features:
    invalid = ((df[feature] < 0) | (df[feature] > 1)).sum()
    print(f"{feature}: {invalid:,} invalid values")

In [ ]:
# check physical constraints
print("Negative durations:", (df["duration_ms"] < 0).sum())
print("Non-positive tempos:", (df["tempo"] <= 0).sum())

In [ ]:
# audit summary
audit = {
    "rows": len(df),
    "columns": df.shape[1],
    "duplicate_rows": df.duplicated().sum(),
    "duplicate_track_ids": df["track_id"].duplicated().sum(),
    "total_missing_values": df.isna().sum().sum(),
    "non_numeric_audio_features": len(non_numeric)
}

pd.Series(audit)

In [ ]:
# feature matrix
X = df[audio_features].copy()
print(f"Feature matrix dimensions: {X.shape}")

In [ ]:
# descriptive stats
feature_stats = X.describe().T

feature_stats["median"] = X.median()
feature_stats["skewness"] = X.skew()

feature_stats = feature_stats[
    ["count", "mean", "std", "min", "25%", "median", "50%", "75%", "max", "skewness"]
]

feature_stats

In [ ]:
# relative skewness ranking
skewness = (
    X.skew()
    .sort_values(key=lambda x: x.abs(), ascending=False)
)

skewness

In [ ]:
# visualize distribution
X.hist(
    bins=30,
    figsize=(16, 12)
)

plt.suptitle("Distributions of Musical Features", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# feature scale comparison
feature_scale = pd.DataFrame({
    "mean": X.mean(),
    "std": X.std(),
    "range": X.max() - X.min(),
    "min": X.min(),
    "max": X.max()
})

feature_scale.sort_values("std", ascending=False)

In [ ]:
# correlation matrix
corr = X.corr()
plt.figure(figsize=(12,10))

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Between Musical Features")
plt.tight_layout()
plt.show()

In [ ]:
# strongest correlations search
upper = corr.where(
    np.triu(
        np.ones(corr.shape),
        k=1
    ).astype(bool)
)

strongest_correlations = (
    upper
    .stack()
    .sort_values(
        key=lambda x: x.abs(),
        ascending=False
    )
)

strongest_correlations.head(10)

In [ ]:
# tabular correlation data
top_correlations = strongest_correlations.head(10).reset_index()

top_correlations.columns = [
    "feature_1",
    "feature_2",
    "correlation"
]

top_correlations["absolute_correlation"] = (
    top_correlations["correlation"].abs()
)

top_correlations

In [ ]:
# strongest correlations visualized
top_5 = top_correlations.head(5)

for _, row in top_5.iterrows():
    feature_1 = row["feature_1"]
    feature_2 = row["feature_2"]
    correlation = row["correlation"]

    plt.figure(figsize=(7, 5))

    sns.scatterplot(
        data=df,
        x=feature_1,
        y=feature_2,
        alpha=0.2
    )

    plt.title(
        f"{feature_1} vs {feature_2} "
        f"(r = {correlation:.2f})"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# highly correlated pair identification
high_corr = strongest_correlations[
    strongest_correlations.abs() >= 0.50
]

high_corr

In [ ]:
# reusable feature summary
feature_summary = pd.DataFrame({
    "mean": X.mean(),
    "std": X.std(),
    "min": X.min(),
    "median": X.median(),
    "max": X.max(),
    "skewness": X.skew()
})

feature_summary

In [ ]:
# save derived summary
feature_summary.to_csv(
    "../data/processed/feature_summary.csv"
)